# MongoDB Handling — PAMAP2 Ingestion

After installing the MongoDB server in your machine, you can use this notebook for handling the initial processes with the database.

Specifically, in this step, we utilize Python's `pymongo` library to exploit its capabilities for MongoDB server interaction.

The wearable hardware that the project was originally built around is not available for this offering of the course, so you will work with the PAMAP2 Physical Activity Monitoring dataset (bundled in `data/PAMAP2_Dataset/`). Treat this notebook as the equivalent of the original "data collection" step: parse the raw `.dat` files, transform each (subject, activity) segment into a MongoDB document, and load it into your collection. The rest of the project (`aiot_project_*.ipynb`) reads from MongoDB, not from the raw files.

**Important Note:** Be sure that the MongoDB server is up and running as a service in the background.

On Windows, MongoDB Community Server runs as a Windows service by default. Verify with PowerShell: `Get-Service MongoDB`. On macOS use `brew services start mongodb-community`. Install instructions: <https://www.mongodb.com/docs/manual/administration/install-community/>.

**Note:** You can modify any of the processes below, however, you have to explain your thoughts.

In [1]:
# import library for various processes with the OS
import os

## Load configuration

In [2]:
# import library for yaml handling
import yaml

In [3]:
config_path = os.path.join(os.getcwd(), "config.yml")

with open(config_path) as file:
    config = yaml.load(file, Loader=yaml.FullLoader)

## MongoDB database instantiation

The relevant information for the MongoDB client connection, the database name, and collection name is located in the configuration file:

```yaml
# DB Connection with the uri (host)
client: "mongodb://localhost:27017/"

# db name
db: "aiot_course"

# db collection
col: "pamap2"
```

In [4]:
# import library for handling the MongoDB client
import pymongo
# import library for retrieving datetime
from datetime import datetime

### Create the database

To create a database in MongoDB, start by creating a `MongoClient` object, then specify a connection URL with the correct IP address and the name of the database you want to create.

MongoDB will create the database if it does not exist, and make a connection to it.

In [5]:
client = pymongo.MongoClient(config["client"])

In [6]:
db = client[config["db"]]

### Instantiate the collection

To create a collection in MongoDB, use the database object and specify the name of the collection you want to create. MongoDB will create the collection if it does not exist.

Initially, no collection will be shown in MongoDB before you enter the first document!

In [7]:
col = db[config["col"]]

## Create the data collection

You will populate the MongoDB collection from the PAMAP2 dataset under `data/PAMAP2_Dataset/`. The directory layout is fixed by the dataset and **must not be renamed or restructured**:

```
data/
└── PAMAP2_Dataset/
    ├── readme.pdf                 (canonical dataset description — read this first)
    ├── DataCollectionProtocol.pdf
    ├── DescriptionOfActivities.pdf
    ├── PerformedActivitiesSummary.pdf
    ├── subjectInformation.pdf
    ├── Protocol/                  (9 subjects, all required activities)
    │   ├── subject101.dat
    │   ├── ...
    │   └── subject109.dat
    └── Optional/                  (5 subjects, additional activities)
        ├── subject101.dat
        ├── ...
        └── subject109.dat
```

Each `.dat` file is a whitespace-separated text file with **54 columns** sampled at **100 Hz**. The full column layout (timestamp, activity_id, heart_rate, then 17 columns per IMU for hand/chest/ankle) is documented in `data/README.md` and in `PAMAP2_Dataset/readme.pdf`.

Before inserting documents, your parsing code must:

- Group the rows into **contiguous (subject, activity) segments** — a subject may perform the same activity in more than one block, and each block should become its own document.
- Drop rows where `activity_id == 0` (transient periods between activities).
- Use only the channels mandated by the "Sensor Selection Strategy" — ±16 g accelerometer + gyroscope per IMU location.
- Ensure the working sampling rate is **100 Hz**. Downsample any channel that is not natively at 100 Hz.

In [8]:
# import library for handling the .dat data and transformations
import pandas as pd
import numpy as np

Get dataset path:

In [9]:
data_path = os.path.join(os.getcwd(), config["data_path"])
print(data_path)

F:\ΣΧΟΛΗ\PROJECTS\IOT\data/PAMAP2_Dataset


List all `.dat` files for each split (`Protocol`, `Optional`):

In [10]:
splits = [s for s in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, s))]
print(splits)

['Protocol']


In [11]:
# print files in a split
split_path = os.path.join(data_path, "Protocol")
files_in_split = sorted(f for f in os.listdir(split_path)
                        if f.endswith(".dat") and os.path.isfile(os.path.join(split_path, f)))
print(files_in_split)

['subject101.dat', 'subject102.dat', 'subject103.dat', 'subject104.dat', 'subject105.dat', 'subject106.dat', 'subject107.dat', 'subject108.dat', 'subject109.dat']


Each document in the MongoDB database should have the following schema (default configuration: hand/wrist IMU, accelerometer + gyroscope):

```json
{
  "_id": ObjectId("6984b3fa87abe7f4dff571aa"),
  "data": {
    "acc_x": ["array", "of", "values"],
    "acc_y": ["array", "of", "values"],
    "acc_z": ["array", "of", "values"],
    "gyr_x": ["array", "of", "values"],
    "gyr_y": ["array", "of", "values"],
    "gyr_z": ["array", "of", "values"]
  },
  "activity_id": 4,
  "activity_label": "walking",
  "subject": "101",
  "split": "Protocol",
  "imu_location": "hand",
  "sensor": "AccGyr",
  "sr": 100,
  "datetime": "MongoDB datetime object (it can be generated with the datetime.datetime.now() function)"
}
```

**Field notes:**
- `activity_id` is the integer label as it appears in the `.dat` file. `activity_label` is the human-readable name.
- `subject` is the file's subject identifier (e.g. `"101"` … `"109"`).
- `split` is `"Protocol"` or `"Optional"`, matching the source subdirectory.
- `imu_location` is `"hand"`, `"chest"`, or `"ankle"`. **Use one document per IMU location**; do not concatenate IMUs into a single document.
- `sensor` is `"Acc"`, `"Gyr"`, or `"AccGyr"`. If we later expand to the magnetometer, use `"AccGyrMag"` and add `mag_x`, `mag_y`, `mag_z` keys with the same axis convention.
- `sr` (sampling rate) must be 100. Downsample any channel that is not natively at 100 Hz before inserting.

**Note:** the document is mandatory to have the aforementioned schema, in order to argue and proceed with the rest of the processes later on, in data engineering, plotting, etc.

In [12]:
from utils import df_rebase, silent_rebase

### Provide the code to parse the PAMAP2 `.dat` files and upload the documents to MongoDB

First, declare which columns of the PAMAP2 `.dat` files we will actually read, the names we will give them, and the mapping from `activity_id` to a human-readable `activity_label`.

The PAMAP2 layout is fixed: 3 meta columns (`timestamp`, `activity_id`, `heart_rate`) followed by 17 columns per IMU (`hand`, `chest`, `ankle`) — `temp`, ±16 g accelerometer (xyz), ±6 g accelerometer (xyz), gyroscope (xyz), magnetometer (xyz), and orientation quaternion (4 entries) — 54 columns in total.

We only need the channels mandated by the Sensor Selection Strategy: `timestamp`, `activity_id`, and, per IMU, the ±16 g accelerometer and gyroscope axes (6 channels × 3 IMUs = 18). That is **20 columns out of 54**. Heart rate, ±6 g accelerometer, magnetometer, temperature and orientation are skipped — `USECOLS` lists exactly the column indices we keep, and `pd.read_csv` is told via `usecols=USECOLS` not to materialise the rest at all (smaller DataFrame, faster read).

In [13]:
# Column indices we read from PAMAP2's 54-column layout — everything else (heart rate,
# ±6 g accelerometer, magnetometer, temperature, orientation quaternion) is skipped.
USECOLS = [
    0,  1,                       # timestamp, activity_id
    4,  5,  6,   10, 11, 12,     # hand:  acc16 + gyr
    21, 22, 23,  27, 28, 29,     # chest: acc16 + gyr
    38, 39, 40,  44, 45, 46,     # ankle: acc16 + gyr
]

COLS = [
    'timestamp', 'activity_id',
    'hand_acc16_x',  'hand_acc16_y',  'hand_acc16_z',
    'hand_gyr_x',    'hand_gyr_y',    'hand_gyr_z',
    'chest_acc16_x', 'chest_acc16_y', 'chest_acc16_z',
    'chest_gyr_x',   'chest_gyr_y',   'chest_gyr_z',
    'ankle_acc16_x', 'ankle_acc16_y', 'ankle_acc16_z',
    'ankle_gyr_x',   'ankle_gyr_y',   'ankle_gyr_z',
]

ACTIVITY = {
    1:  'lying',           2:  'sitting',            3:  'standing',
    4:  'walking',         5:  'running',            6:  'cycling',
    7:  'nordic_walking',  12: 'ascending_stairs',   13: 'descending_stairs',
    16: 'vacuum_cleaning', 17: 'ironing',            24: 'rope_jumping',
}

print(f'Columns we keep:     {len(COLS)} (out of 54)')
print(f'Protocol activities: {len(ACTIVITY)}')

Columns we keep:     20 (out of 54)
Protocol activities: 12


For every `.dat` file under `Protocol/`, the loop below:

1. **Reads** the file with `pandas`, naming the 54 columns with `COLS`.
2. **Handles NaN** with linear interpolation along each column (justified: IMU/heart-rate channels are continuous signals, so a linear estimate between two known samples is the closest approximation to the true value), then drops any residual leading NaN that has no earlier value to interpolate from.
3. **Drops the transient periods** (`activity_id == 0`).
4. **Detects contiguous activity segments** — every time `activity_id` changes, a new segment starts. Each segment becomes one (or three) MongoDB document(s).
5. For each IMU location (`hand`, `chest`, `ankle`), uses **`df_rebase`** from `utils` to select the ±16 g accelerometer and gyroscope columns of that location and rename them to the canonical `acc_x/y/z`, `gyr_x/y/z` keys required by the document schema.
6. Emits **one document per (segment, IMU location)** following the schema above.

Heart-rate, magnetometer, ±6 g accelerometer, temperature and orientation channels are intentionally **not** included, per the Sensor Selection Strategy of the spec.

Sampling rate is `100 Hz` natively for every channel we keep, so no down-sampling is needed.

In [14]:
# Start from an empty collection — re-running this cell should produce the same documents,
# not duplicate them.
col.drop()

split = 'Protocol'
split_path = os.path.join(data_path, split)

# Canonical schema-side names for the 6 sensor channels inside each document's `data` object
REF = ['acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z']

for filename in sorted(os.listdir(split_path)):
    if not filename.endswith('.dat'):
        continue

    subject = filename.replace('subject', '').replace('.dat', '')

    # 1. Read only the columns we actually need (USECOLS), naming them per COLS
    df = pd.read_csv(os.path.join(split_path, filename),
                     sep=' ', header=None, usecols=USECOLS, names=COLS)

    # 2. Linear interpolation for NaN, then drop any residual leading NaN
    df = df.interpolate(method='linear').dropna()

    # 3. Drop transient periods (activity_id == 0)
    df = df[df['activity_id'] != 0].reset_index(drop=True)

    # 4. Detect contiguous activity segments
    df['segment'] = (df['activity_id'].diff() != 0).cumsum()

    # 5. Per IMU location: rebase to canonical names, then 6. emit one doc per segment
    docs = []
    for imu in ['hand', 'chest', 'ankle']:
        target = [f'{imu}_acc16_x', f'{imu}_acc16_y', f'{imu}_acc16_z',
                  f'{imu}_gyr_x',   f'{imu}_gyr_y',   f'{imu}_gyr_z',
                  'activity_id', 'segment']
        ref = REF + ['activity_id', 'segment']
        view = silent_rebase(df, target, ref)

        for _, seg in view.groupby('segment'):
            activity_id = int(seg['activity_id'].iloc[0])
            docs.append({
                'data':           {k: seg[k].tolist() for k in REF},
                'activity_id':    activity_id,
                'activity_label': ACTIVITY[activity_id],
                'subject':        subject,
                'split':          split,
                'imu_location':   imu,
                'sensor':         'AccGyr',
                'sr':             100,
                'datetime':       datetime.now(),
            })

    col.insert_many(docs)
    print(f'subject {subject}: inserted {len(docs)} documents')

print(f'\nTotal documents in collection: {col.count_documents({})}')

subject 101: inserted 42 documents


subject 102: inserted 42 documents


subject 103: inserted 33 documents


subject 104: inserted 39 documents


subject 105: inserted 42 documents


subject 106: inserted 42 documents


subject 107: inserted 39 documents


subject 108: inserted 42 documents
subject 109: inserted 3 documents



Total documents in collection: 324
